# Create a wavelength lookup table for DREAM

This notebook shows how to create a wavelength lookup table for the DREAM instrument.

In [ ]:
import scipp as sc
from ess.reduce import kinematics as kin
from ess.reduce.nexus.types import AnyRun
from ess.dream.beamline import InstrumentConfiguration, choppers

## Select the choppers

We select the choppers for the 'high-flux' configuration.

In [ ]:
disk_choppers = choppers(InstrumentConfiguration.high_flux_BC215)

Note that possible configurations are `high_flux_BC215`, `high_flux_BC240`, and `high_resolution`.

## Setting up the workflow

In [ ]:
wf = kin.LookupTableWorkflow()

wf[kin.LtotalRange] = sc.scalar(5.0, unit="m"), sc.scalar(80.0, unit="m")
wf[kin.NumberOfSimulatedNeutrons] = 5_000_000  # Increase this number for more reliable results
wf[kin.SourcePosition] = sc.vector([0, 0, 0], unit='m')
wf[kin.DiskChoppers[AnyRun]] = disk_choppers
wf[kin.DistanceResolution] = sc.scalar(0.1, unit="m")
wf[kin.TimeResolution] = sc.scalar(250.0, unit='us')
wf[kin.PulsePeriod] = 1.0 / sc.scalar(14.0, unit="Hz")
wf[kin.PulseStride] = 1
wf[kin.PulseStrideOffset] = None

## Compute the table

In [ ]:
table = wf.compute(kin.LookupTable)
table.array

In [ ]:
table.plot()

## Save to file

In [ ]:
table.save_hdf5('DREAM-high-flux-wavelength-lut-5m-80m-bc215.h5')